In [ ]:
import os
base_directory = os.path.dirname(os.path.abspath(""))

import pandas as pd
from radiology_dataset_db.extract_radiology_dataset_information_llm import extract_radiology_dataset_info_with_agent

# import importlib
# import radiology_dataset_db.extract_radiology_dataset_information_llm as erdil
# importlib.reload(erdil)
# from radiology_dataset_db.extract_radiology_dataset_information_llm import extract_radiology_dataset_info_with_agent

gpu_id = 0
vllm_port = 8001
model = "Qwen/Qwen2.5-7B-Instruct"

# Setting up vLLM

In [2]:
# check if vllm running in port {vllm_port}, if not, start it
import socket
def is_port_in_use(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(('localhost', port)) == 0

if not is_port_in_use(vllm_port):
    !conda run -n vllm CUDA_VISIBLE_DEVICES={gpu_id} python -m vllm.entrypoints.openai.api_server --model {model} --port {vllm_port} --enforce-eager --enable-auto-tool-choice --tool-call-parser hermes > vllm.log 2>&1 &

# Test run

In [ ]:
title = "Merlin: a computed tomography vision–language foundation model and dataset"
abstract = "The large volume of abdominal computed tomography (CT) scans1,2 coupled with the shortage of radiologists3,4,5,6 have intensified the need for automated medical image analysis tools. Previous state-of-the-art approaches for automated analysis leverage vision–language models (VLMs) that jointly model images and radiology reports7,8,9,10,11,12. However, current medical VLMs are generally limited to 2D images and short reports. Here to overcome these shortcomings for abdominal CT interpretation, we introduce Merlin, a 3D VLM that learns from volumetric CT scans, electronic health record data and radiology reports. This approach is enabled by a multistage pretraining framework that does not require additional manual annotations. We trained Merlin using a high-quality clinical dataset of paired CT scans (>6 million images from 15,331 CT scans), diagnosis codes (>1.8 million codes) and radiology reports (>6 million tokens). We comprehensively evaluated Merlin on 6 task types and 752 individual tasks that covered diagnostic, prognostic and quality-related tasks. The non-adapted (off-the-shelf) tasks included zero-shot classification of findings (30 findings), phenotype classification (692 phenotypes) and zero-shot cross-modal retrieval (image-to-findings and image-to-impression). The model-adapted tasks included 5-year chronic disease prediction (6 diseases), radiology report generation and 3D semantic segmentation (20 organs). We validated Merlin at scale, with internal testing on 5,137 CT scans and external testing on 44,098 CT scans from 3 independent sites and 2 public datasets. The results demonstrated high generalization across institutions and anatomies. Merlin outperformed 2D VLMs, CT foundation models and off-the-shelf radiology models. We also computed scaling laws and conducted ablation studies to identify optimal training strategies. We release our trained models, code and dataset for 25,494 pairs of abdominal CT scans and radiology reports. Our results demonstrate how Merlin may assist in the interpretation of abdominal CT scans and mitigate the burden on radiologists while simultaneously adding value for future biomarker discovery and disease risk stratification."
link = "https://doi.org/10.1038/s41586-026-10181-8"

dataset = await extract_radiology_dataset_info_with_agent(title, abstract, publication_metadata={"link": link})
print(dataset)

2026-03-25 01:24:01,243 DEBUG | LLM classified as dataset paper.
2026-03-25 01:24:03,547 DEBUG | LLM output: name='Merlin' num_images=6000000 num_patients=15331 modalities=[<Modality.CT: 'CT'>] body_regions=[<BodyRegion.ABDOMEN: 'abdomen'>] additional_data=[<AdditionalData.REPORTS: 'reports'>] paper_title='Merlin: a computed tomography vision–language foundation model and dataset' paper_link=None paper_year=None paper_authors=[] paper_journal=None pmid=None paper_citation_count=None


name='Merlin' num_images=6000000 num_patients=15331 modalities=[<Modality.CT: 'CT'>] body_regions=[<BodyRegion.ABDOMEN: 'abdomen'>] additional_data=[<AdditionalData.REPORTS: 'reports'>] paper_title='Merlin: a computed tomography vision–language foundation model and dataset' paper_link='https://doi.org/10.1038/s41586-026-10181-8' paper_year=None paper_authors=None paper_journal=None pmid=None paper_citation_count=None


# Run the script with a few papers

In [ ]:
max_papers = 10
output_path = "output/radiology_db_notebook.csv"

!python3 {base_directory}/scripts/build_db.py --max-papers {max_papers} --output-path {output_path}

2026-03-25 01:24:04,912 INFO | Overriding config: output_path = radiology_db_notebook.csv
2026-03-25 01:24:04,912 INFO | Overriding config: max_papers = 10
2026-03-25 01:24:04,912 INFO | No existing output file found at radiology_db_notebook.csv. A new file will be created.
2026-03-25 01:24:04,912 INFO | Using model: openai:Qwen/Qwen2.5-7B-Instruct
2026-03-25 01:24:04,912 INFO | Using Entrez API key
2026-03-25 01:24:04,912 INFO | Searching PubMed...
2026-03-25 01:24:04,912 DEBUG | PubMed query: ("Database Management Systems"[MeSH] OR dataset[ti] OR database[ti] OR "data collection"[ti] OR "information repository"[ti] OR benchmark[ti] OR "challenge data"[ti] OR "data commons"[ti] OR "data repository"[ti] OR "data sharing"[ti]) AND ("Radiology"[MeSH] OR "Radiography"[MeSH] OR "Radiology Information Systems"[MeSH] OR radiology[tiab] OR radiograph[tiab] OR "Diagnostic Imaging"[tiab] OR "Medical Image"[tiab] OR "Medical Imaging"[tiab] OR "Biomedical Image"[tiab] OR "Biomedical Imaging"[tiab

# End the vLLM server

In [5]:
# check if it's running
if is_port_in_use(vllm_port):
    print(f"vLLM is running on port {vllm_port}")
    !pkill -f vllm

vLLM is running on port 8001


# View the df

In [ ]:
df = pd.read_csv(output_path)
df.head()

,name,num_images,num_patients,modalities,body_regions,additional_data,paper_title,paper_link,paper_year,paper_authors,paper_journal,pmid,paper_citation_count
0,A preclinical CT and MRI Liver Imaging Dataset...,222.0,NaN,"CT,MRI",abdomen,segmentation,A preclinical CT and MRI Liver Imaging Dataset...,https://doi.org/10.1038/s41597-026-07003-x,2026,"Schraven S,Gonzalez C,Baskaya F,Krott L,Becker...",Scientific data,41872225,0
1,Ethmoid CBCT imaging dataset,565.0,565.0,CT,limbs,segmentation,Ethmoid sinus CBCT imaging as a biometric inst...,https://doi.org/10.1016/j.ejrad.2026.112796,2026,"Alsalama A,Alsmirat M,Al-Rawi N,Uthman A,Elnag...",European journal of radiology,41864172,0
2,Brain Metastases Dataset,NaN,20.0,"CT,MRI",brain,NaN,Expert variability as a benchmark for validati...,https://doi.org/10.1016/j.ejmp.2026.105765,2026,"Faccenda V,Panizza D,Pinzi V,Carminati S,Colci...",Physica medica : PM : an international journal...,41855698,0
3,MHS Data Repository,NaN,9500000.0,NaN,NaN,"reports,reports,reports,reports,reports",The Military Health System Data Repository: Le...,https://doi.org/10.1002/pds.70338,2026,"Day WG,Reynolds MW,Pittman L,Edison J,Robins R...",Pharmacoepidemiology and drug safety,41852121,0
4,DyABD,NaN,NaN,MRI,abdomen,segmentation,DyABD: the abdominal muscle segmentation in dy...,https://doi.org/10.1186/s12880-026-02204-7,2026,"Belton N,Joppin V,Lawlor A,Masson C,Bege T,Ben...",BMC medical imaging,41851881,0


In [7]:
!pip list

Package                                  Version         Build Editable project location
---------------------------------------- --------------- ----- -----------------------------------------------
accelerate                               1.13.0
ag-ui-protocol                           0.1.14
aiofile                                  3.9.0
aiohappyeyeballs                         2.6.1
aiohttp                                  3.13.3
aiosignal                                1.4.0
annotated-doc                            0.0.4
annotated-types                          0.7.0
anthropic                                0.86.0
anyio                                    4.12.1
apache-tvm-ffi                           0.1.9
argcomplete                              3.6.3
astor                                    0.8.1
asttokens                                3.0.1
async-timeout                            5.0.1
attrs                                    26.1.0
Authlib                                  1